# Gap 4: Hard Nesting-Depth Threshold $N_k$

All previous tests used *soft* attenuation $N_\lambda$ (continuous weighting).
This gap tests the *threshold* operator $N_k$ from the foundations paper:

$$N_k(x) = \begin{cases} x & \sigma(x) < k \\ 0 & \sigma(x) \geq k \end{cases}$$

Applied to nesting depth: diagrams with $\geq k$ levels of nested self-energy
insertions are hard-killed (zeroed), others survive at full weight.

- $k = 1$ (binary SCN): kill ALL self-energy insertions
- $k = 2$: kill doubly-nested and deeper
- $k \to \infty$: standard (no threshold)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.graded_scn import (
    ALPHA_S_MZ, M_Z, M_TAU,
    beta0_standard, beta0_attenuated, alpha_s_running_1loop,
    r_tau_coefficients, r_tau_tree,
    R_TAU_EXPERIMENT, R_TAU_UNCERTAINTY,
    BETA0_GLUON_LOOP, BETA0_GHOST_LOOP, BETA0_QUARK_PER_NF,
    ALPHA_OVER_PI, A_E_EXPERIMENT, A_E_UNCERTAINTY,
    QED_G2_COEFFICIENTS, compute_g2_standard,
)

nf = 3
b0_std = beta0_standard(nf)

print("=== beta_0 Under Hard Nesting Threshold ===")
print()
print("beta_0 decomposition by nesting depth:")
print(f"  Gluon loop (sigma_nest=1): {BETA0_GLUON_LOOP:+.1f}")
print(f"  Ghost loop (sigma_nest=0): {BETA0_GHOST_LOOP:+.1f}")
print(f"  Quark loops (sigma_nest=0): {BETA0_QUARK_PER_NF*nf:+.1f} (nf={nf})")
print(f"  Total beta_0 = {b0_std:.1f}")
print()

# Threshold at nesting depth k:
# k=inf: standard; k=2: same at 1-loop; k=1: binary SCN
thresholds = [
    ("k=inf (standard)", b0_std),
    ("k=2 (depth>=2 killed)", b0_std),  # same at 1-loop
    ("k=1 (binary SCN)", BETA0_GHOST_LOOP + BETA0_QUARK_PER_NF * nf),
]

print("beta_0 values under threshold:")
for label, b0 in thresholds:
    alpha_s = alpha_s_running_1loop(M_TAU**2, nf, b0)
    af = "AF" if b0 > 0 else "NO AF"
    if np.isnan(alpha_s):
        print(f"  {label}: beta_0={b0:.1f} ({af}) -> Landau pole!")
    else:
        print(f"  {label}: beta_0={b0:.1f} ({af}) -> alpha_s(m_tau)={alpha_s:.4f}")

print()
print("At 1-loop, only k=1 changes beta_0. k>=2 has no effect because")
print("the deepest nesting in the 1-loop gluon SE is depth 1.")
print()
print("Binary SCN (k=1): beta_0 = -1 -> asymptotic freedom LOST.")
print("This was already falsified at >= 415 sigma.")

In [ ]:
# K_n threshold: kill bubble-chain contributions at nesting depth >= k

K = r_tau_coefficients()
alpha_s_mtau = alpha_s_running_1loop(M_TAU**2, nf, b0_std)
a_s = alpha_s_mtau / np.pi

def r_tau_threshold(f, k_thresh, max_order=4):
    """
    R_tau with hard nesting-depth threshold on bubble chains.
    
    Bubble chain in K_n has max nesting depth (n-1).
    If (n-1) >= k_thresh: bubble part killed.
    Non-bubble part always survives.
    """
    delta = K[1] * a_s  # n=1: depth 0, always survives
    for n in range(2, max_order + 1):
        depth = n - 1
        if depth >= k_thresh:
            k_eff = (1 - f) * K[n]  # kill bubble part
        else:
            k_eff = K[n]  # below threshold: full K_n
        delta += k_eff * a_s**n
    return r_tau_tree() * (1 + delta)

print(f"alpha_s(m_tau) = {alpha_s_mtau:.4f}, a_s = {a_s:.5f}")
print()

# Scan over threshold and bubble fraction
print("R_tau under nesting-depth threshold:")
print(f"{'k':>8} {'f=0.0':>10} {'f=0.3':>10} {'f=0.5':>10} {'f=0.7':>10} {'f=1.0':>10}")
for k in [1, 2, 3, 4, 100]:
    k_label = 'inf' if k >= 100 else str(k)
    row = f"{k_label:>8}"
    for f in [0.0, 0.3, 0.5, 0.7, 1.0]:
        r = r_tau_threshold(f, k)
        row += f" {r:10.4f}"
    print(row)

print(f"\nExperiment: {R_TAU_EXPERIMENT:.4f} +/- {R_TAU_UNCERTAINTY:.4f}")
r_std = r_tau_threshold(0, 100)
print(f"Standard:   {r_std:.4f} ({(r_std-R_TAU_EXPERIMENT)/R_TAU_UNCERTAINTY:+.1f} sigma)")

# Sigma deviations
print("\nSigma deviations from experiment:")
print(f"{'k':>8} {'f=0.3':>10} {'f=0.5':>10} {'f=0.7':>10} {'f=1.0':>10}")
for k in [1, 2, 3, 4]:
    k_label = str(k)
    row = f"{k_label:>8}"
    for f in [0.3, 0.5, 0.7, 1.0]:
        r = r_tau_threshold(f, k)
        sig = (r - R_TAU_EXPERIMENT) / R_TAU_UNCERTAINTY
        row += f" {sig:+10.1f}"
    print(row)

In [ ]:
# Visualize: R_tau vs bubble fraction f, at different k thresholds

f_vals = np.linspace(0, 1, 200)

plt.figure(figsize=(10, 6))
for k in [1, 2, 3, 4]:
    r_vals = [r_tau_threshold(f, k) for f in f_vals]
    plt.plot(f_vals, r_vals, lw=2, label=f'k={k}')

plt.axhline(R_TAU_EXPERIMENT, color='red', ls='--', lw=1.5, label=f'Experiment ({R_TAU_EXPERIMENT})')
plt.fill_between(f_vals, R_TAU_EXPERIMENT - R_TAU_UNCERTAINTY,
                 R_TAU_EXPERIMENT + R_TAU_UNCERTAINTY, alpha=0.2, color='red')
plt.xlabel('Bubble-chain fraction f')
plt.ylabel('R_tau')
plt.title('R_tau under hard nesting threshold at various depths')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find f values where each k matches experiment
print("Bubble fraction f needed to match R_tau at each threshold k:")
for k in [1, 2, 3, 4]:
    f_match = None
    for f in np.linspace(0, 1, 10000):
        r = r_tau_threshold(f, k)
        if abs(r - R_TAU_EXPERIMENT) < R_TAU_UNCERTAINTY:
            f_match = f
            break
    if f_match is not None:
        print(f"  k={k}: f >= {f_match:.3f}")
    else:
        print(f"  k={k}: no f in [0,1] matches (threshold too weak)")

In [ ]:
# QED: threshold on 2-loop nesting
SE_TOTAL = 0.7714
C2_TOTAL_QED = QED_G2_COEFFICIENTS[2]["value"]
SKEL_TOTAL = C2_TOTAL_QED - SE_TOTAL

print("=== QED 2-Loop Threshold ===")
print()

def a_e_threshold_2loop(kill_se=False, max_loop=5):
    total = 0.0
    for n in range(1, max_loop + 1):
        if n == 2 and kill_se:
            coeff = SKEL_TOTAL
        else:
            coeff = QED_G2_COEFFICIENTS[n]["value"]
        total += coeff * ALPHA_OVER_PI**n
    return total

a_std = compute_g2_standard(5)
a_kill = a_e_threshold_2loop(kill_se=True)

print(f"Standard a_e:    {a_std:.15e}")
print(f"Kill-SE a_e:     {a_kill:.15e}")
print(f"Experiment:      {A_E_EXPERIMENT:.15e}")
print(f"Delta(kill-std): {a_kill - a_std:.4e}")
print(f"Sigma deviation: {(a_kill - A_E_EXPERIMENT)/A_E_UNCERTAINTY:.0f} sigma")
print()
print("This IS binary SCN at 2-loop. Already falsified at >= 415 sigma.")
print()
print("Summary:")
print("- k=1 on beta_0: kills asymptotic freedom (beta_0 < 0)")
print("- k=1 on 2-loop QED: 415+ sigma deviation")
print("- k>=2: no effect on 1-loop beta_0 or 2-loop QED")
print("        (deepest nesting is 1 at these orders)")
print("- k>=2 on K_n: selective truncation of model-dependent")
print("  bubble-chain fraction, requires unknown f parameter")
print()
print("VERDICT: Hard threshold is either too destructive (k=1)")
print("or requires unknown parameters (k>=2).")

## Conclusions

The threshold operator $N_k$ applied at nesting depth $k$:

1. **$k=1$ (binary SCN)**: Destroys asymptotic freedom ($\beta_0 \to -1$) and
   deviates from QED $g-2$ at $\geq 415\sigma$. Already falsified.

2. **$k \geq 2$**: Has no effect on 1-loop $\beta_0$ (deepest nesting is 1).
   For $K_n$ coefficients, it selectively truncates the bubble-chain fraction,
   but this requires an unknown parameter $f$ — curve fitting, not prediction.

3. **QED**: Threshold at 2-loop kills SE insertions (= binary SCN). No testable
   threshold exists at $k \geq 2$ because the maximum nesting depth at 2-loop is 1.

**Verdict**: Hard threshold is too aggressive at $k=1$ and indistinguishable
from standard QFT at $k \geq 2$ (given available perturbative data).